# Round — AI-facilitated Scripture reading circles**Scripture in New Frontiers — Gloo AI + YouVersion, Kaggle 2026**| | ||---|---|| **Live project** | https://yv-gloo-chalange.vercel.app — no account needed, tap **Instant Access** || **Video** | https://www.youtube.com/watch?v=YzcBW4xfzeM || **Code** | https://github.com/radoslawkrolikowski/YVGlooChalange |## The frontier**Social platforms — Scripture as conversation, not broadcast.**Round is not a Bible reader. YouVersion already built the best one. Round is the*facilitation layer* that surrounds the reading: a circle of three to five peopleon the same plan, opened and kept alive by a team of specialised AI agents. Thesame relationship Strava has to running.The problem it addresses is not access to Scripture. It is that reading alone iseasy to start and easy to abandon — you hit a wall in the text and there isnobody to ask.## What this notebook isThe app is a Next.js + TypeScript product; this notebook is its **verificationsurface**. Every cell below makes a **live call to the same endpoints, with thesame request shapes, as the deployed app**, and points at the exact TypeScriptmodule that owns that call in production. Nothing here is mocked.It also documents the three non-obvious engineering findings that shaped thebuild — refusal detection, filename-scoped retrieval, and audit-by-constructionlogging — each with the code that acts on them.

## Architecture```                         ┌──────────────────────────────┐   Browser ─────────────►│  Next.js 16 App Router        │   (no account needed)   │  Server Components + Actions  │                         └───────────┬──────────────────┘                                     │              ┌──────────────────────┴────────────────────────┐              │                                               │   ┌──────────▼───────────┐                      ┌────────────▼─────────────┐   │  src/lib/youversion  │                      │     src/lib/gloo.ts      │   │  (the ONLY module    │                      │  (the ONLY LLM gateway   │   │   touching the       │                      │   in the codebase)       │   │   YouVersion SDK)    │                      │                          │   └──────────┬───────────┘                      └────────────┬─────────────┘              │                                               │   api.youversion.com                            platform.ai.gloo.com   · /v1/bibles                                  · /oauth2/token   · /v1/bibles/{id}/passages/{usfm}             · /ai/v2/chat/completions   · /v1/highlights                              · /ai/v2/chat/completions/grounded   · OAuth (Sign in with YouVersion)             · /ai/v1/data/search              │                                               │              └──────────────────┬────────────────────────────┘                                 │                     ┌───────────▼────────────┐                     │  Postgres (Drizzle)    │                     │  agent_logs ← every    │                     │  model call, ok or not │                     └────────────────────────┘```**Two hard architectural rules, both checkable in one command:**1. *Gloo is the only LLM gateway.* Every model call in the app goes through   `chatCompletion()` in `src/lib/gloo.ts`. No other model provider package   exists in the repo — verify with   `grep -rniE "\b(openai|anthropic|@ai-sdk|mistral|cohere)\b" src package.json`.2. *YouVersion is the only source of Scripture text.* No passage text is stored,   cached long-term, or translated by a model. When a Spanish reader opens a   passage, the text **arrives in Spanish from YouVersion** — it is never an   English verse run through a model.**The agents** (each one a distinct system prompt + Gloo call, all logged):`facilitator` (pre-reading prompts, conversation starters, lesson summaries),`companion` (passage Q&A grounded in commentary), `circle-bot` (seeded AI circlemembers), `prayer` (prayer generation from the reader's own reading),`reminder` (nudges), plus the Escalation screen that runs before anything amember writes reaches the circle.

## Setup**To run this notebook live you need internet enabled** (Notebook settings →Internet → On) **and four credentials** added as Kaggle Secrets (Add-ons →Secrets):| Secret | Where it comes from ||---|---|| `YOUVERSION_API_KEY` | platform.youversion.com app key || `GLOO_CLIENT_ID` | Gloo AI Studio OAuth2 client || `GLOO_CLIENT_SECRET` | Gloo AI Studio OAuth2 client || `GLOO_SEARCH_TENANT` | normalised publisher name for the Data Engine corpus |No credential is printed by any cell, and none is committed to the repo. Cellsdegrade to an explanatory message when a credential is missing, so the notebookstays readable end to end without them.

In [ ]:
import base64, json, os, re, textwrapimport requestsdef secret(name):    """Kaggle Secret, else environment variable, else None. Never prints a value."""    try:        from kaggle_secrets import UserSecretsClient        return UserSecretsClient().get_secret(name)    except Exception:        return os.environ.get(name)YOUVERSION_API_KEY  = secret("YOUVERSION_API_KEY")GLOO_CLIENT_ID      = secret("GLOO_CLIENT_ID")GLOO_CLIENT_SECRET  = secret("GLOO_CLIENT_SECRET")GLOO_SEARCH_TENANT  = secret("GLOO_SEARCH_TENANT")HAS_YV   = bool(YOUVERSION_API_KEY)HAS_GLOO = bool(GLOO_CLIENT_ID and GLOO_CLIENT_SECRET)def live(label, fn):    """Run a live API call; print a clean diagnosis instead of raising.    A published notebook should degrade to an explanation, not a traceback.    """    try:        return fn()    except requests.HTTPError as error:        status = error.response.status_code if error.response is not None else "?"        hint = {            401: "credentials rejected — check the client id/secret pair",            403: "forbidden — for Gloo this is almost always the publisher/tenant "                 "string (display name uses hyphens, tenant uses underscores); "                 "for YouVersion it is an unlicensed Bible version",            404: "not found — check the reference or version id",            429: "rate limited",        }.get(status, "see the response body")        print(f"{label}: HTTP {status} — {hint}")    except Exception as error:        print(f"{label}: {type(error).__name__} — {error}")    return Nonedef show(title, body, width=88):    print(title)    print("-" * len(title))    for para in str(body).split("\n"):        print(textwrap.fill(para, width) if para else "")print("YouVersion credential present:", HAS_YV)print("Gloo credentials present:    ", HAS_GLOO)

---# Part 1 — YouVersion Platform APIProduction owner: [`src/lib/youversion.ts`](https://github.com/radoslawkrolikowski/YVGlooChalange/blob/main/src/lib/youversion.ts)— a thin adapter over the official `@youversion/platform-core` SDK (v2.2.0). It isthe only module in the app that touches the SDK, so SDK churn stays contained toone file. The app key is attached server-side on every request and the env var isdeliberately **not** `NEXT_PUBLIC_`-prefixed, so it can never reach the clientbundle.The SDK is JavaScript, so the cells below call the same REST surface directly:base `https://api.youversion.com`, app key in the `X-YVP-App-Key` header.

In [ ]:
YV_BASE = "https://api.youversion.com"def yv_get(path, params=None):    """One YouVersion GET. Mirrors ApiClient in @youversion/platform-core."""    response = requests.get(        f"{YV_BASE}{path}",        headers={"X-YVP-App-Key": YOUVERSION_API_KEY, "Accept": "application/json"},        params=params or {},        timeout=30,    )    response.raise_for_status()    return response.json()def fetch_passage(reference, version_id, fmt="text"):    """Port of fetchPassage() — GET /v1/bibles/{versionId}/passages/{usfm}.    References are USFM: "JHN.3.16", "PSA.23", "JHN.3.1-5".    The passage payload carries no version metadata, so the app fetches the    version record separately for the abbreviation the UI must always display.    """    passage = yv_get(f"/v1/bibles/{version_id}/passages/{reference}", {"format": fmt})    version = yv_get(f"/v1/bibles/{version_id}")    return {        "reference": passage.get("reference") or reference,        "content": passage.get("content", ""),        "versionId": version_id,        "versionAbbreviation": version.get("abbreviation", ""),    }

### The multilingual claim, testedThe strongest thing Round does with the YouVersion API is also the simplest:**a Spanish reader gets Spanish Scripture from YouVersion, not an English versetranslated by a model.** The two cells below fetch the same passage in twoversions and show the text arriving natively in each language.Version IDs come from `src/config/bible-versions.ts`, where every ID was verifiedlive against the Bible Versions API. An app key can only *fetch* versions whoselicense is enabled for it, so the catalogue in that file records a `licensed`flag per version and the app falls back to a licensed version per language(English → BSB 3034, Spanish → RVES 147, Portuguese → BLT 3254) until therequested licenses are granted. Unlicensed versions return **403 on passagefetch** while still appearing in the catalogue — a distinction worth designingfor rather than discovering in production.

In [ ]:
BSB, RVES = 3034, 147  # licensed for this app key todayif not HAS_YV:    print("No YOUVERSION_API_KEY — skipping live fetch.")else:    for version_id in (BSB, RVES):        passage = fetch_passage("PSA.23.1-3", version_id)        show(f"{passage['reference']}  ({passage['versionAbbreviation']})",             re.sub(r"\s+", " ", passage["content"]).strip())        print()

### Reading plans, highlights, and the rest of the surfaceThe app uses more of the API than one passage fetch. Each of these is a realcall in production:| Round feature | YouVersion call | Module ||---|---|---|| Passage in the reader | `GET /v1/bibles/{id}/passages/{usfm}` | `fetchPassage()` || Version chip + copyright | `GET /v1/bibles/{id}` | `fetchVersion()` || Language/version picker | `GET /v1/bibles` | `listBibleVersions()` || Plan builder reference gate | passage fetch, 404 = invalid | `validatePassageReference()` || Highlight import + circle context | `GET /v1/highlights` | `fetchChapterHighlights()` || Book/chapter navigation | `GET /v1/bibles/{id}/index` | `fetchBibleIndex()` || Sign in with YouVersion | OAuth authorize + token | `src/lib/auth.ts` |`validatePassageReference()` is the one worth calling out. When a user asks for acustom AI-generated plan, Gloo proposes the references — and a model willcheerfully invent `PSA.151.3`. Every proposed reference is fetched againstYouVersion before the plan is written; a 404 is fed back into the regenerationprompt as the failure reason, while non-404 errors (auth, network, 5xx) arere-thrown because they say nothing about the reference itself. **The Bible API isused as the truth oracle over the model's output.**

In [ ]:
def validate_reference(reference, version_id):    """Port of validatePassageReference(): None when valid, message when not.    Cheaper than fetch_passage — validation does not need version metadata.    """    try:        passage = yv_get(f"/v1/bibles/{version_id}/passages/{reference}", {"format": "text"})    except requests.HTTPError as error:        if error.response is not None and error.response.status_code == 404:            return f"{reference} does not resolve in version {version_id}"        raise  # auth / network / 5xx says nothing about the reference    return None if passage.get("content") else f"{reference} resolved but returned no text"if not HAS_YV:    print("No YOUVERSION_API_KEY — skipping live validation.")else:    for reference in ("PSA.23.1", "PSA.151.3"):     # one real, one hallucinated        verdict = validate_reference(reference, BSB)        print(f"{reference:<12} {'VALID' if verdict is None else 'REJECTED — ' + verdict}")

---# Part 2 — Gloo AI Studio APIProduction owner: [`src/lib/gloo.ts`](https://github.com/radoslawkrolikowski/YVGlooChalange/blob/main/src/lib/gloo.ts)— the app's only LLM gateway. It handles the OAuth2 token exchange, retries,guardrail and refusal detection, and writes an `agent_logs` row for **every**call, success or failure, before returning.Auth is OAuth2 client credentials: the client ID and secret are exchanged at thetoken endpoint for a bearer JWT that expires after one hour. Production cachesthe token in module scope and refreshes it 60 seconds early, so a serverlessinstance re-authenticates at most once per hour.Note the implementation choice: the app uses plain `fetch` rather than the OpenAISDK. Completions V2 is OpenAI-shaped (`messages` / `choices` / `usage`) *plus*Gloo-only fields — `auto_routing`, `model_family`, `tradition`, and routingmetadata on the response. Plain HTTP keeps those first-class and keeps any otherprovider's package out of the repo, which is what makes rule 1 verifiable.

In [ ]:
GLOO_TOKEN_URL       = "https://platform.ai.gloo.com/oauth2/token"GLOO_COMPLETIONS_URL = "https://platform.ai.gloo.com/ai/v2/chat/completions"GLOO_GROUNDED_URL    = GLOO_COMPLETIONS_URL + "/grounded"GLOO_SEARCH_URL      = "https://platform.ai.gloo.com/ai/v1/data/search"GLOO_COLLECTION      = "GlooProd"   # the only collection Gloo exposes for Data Engine content_token_cache = {}def gloo_token():    """Port of getAccessToken(): client-credentials exchange, cached, refreshed 60s early."""    import time    now = time.time()    if _token_cache.get("access_token") and now < _token_cache["refresh_after"]:        return _token_cache["access_token"]    basic = base64.b64encode(f"{GLOO_CLIENT_ID}:{GLOO_CLIENT_SECRET}".encode()).decode()    response = requests.post(        GLOO_TOKEN_URL,        headers={            "Content-Type": "application/x-www-form-urlencoded",            "Authorization": f"Basic {basic}",        },        data="grant_type=client_credentials&scope=api/access",        timeout=30,    )    response.raise_for_status()    token = response.json()    _token_cache.update(        access_token=token["access_token"],        refresh_after=now + token["expires_in"] - 60,    )    return token["access_token"]if not HAS_GLOO:    print("No Gloo credentials — skipping token exchange.")else:    token = gloo_token()    print("Token acquired. Length:", len(token), "· expires in ~1h · value never printed.")

### A real agent call — the Facilitator's pre-reading promptsThis is the first AI a Round user meets: before the passage text loads, theFacilitator generates two or three short prompts that give a reader something tocarry into the text. The system prompt below is the shape used in production(`src/lib/pre-reading.ts`).`auto_routing: true` lets Gloo pick the model and report back which one servedthe request; `tradition` is a Gloo-only field that steers the completion toward afaith context. Production passes an explicit `model` only where a specificfamily is needed — and there is a documented trap: Gloo rejects`tradition: "not_faith_specific"` unless a model is named (`422 Model is requiredwhen tradition is set to not_faith_specific`), so `completionBody()` catches thatcontradiction locally, with the reason, instead of letting it surface as anopaque upstream validation error.

In [ ]:
def gloo_completion(messages, model=None, temperature=0.7, max_tokens=400,                    tradition="christian", url=None, extra=None):    """Port of chatCompletion()'s request half — same body shape as production."""    if tradition == "not_faith_specific" and not model:        raise ValueError(            'Gloo requires an explicit model when tradition is "not_faith_specific" — '            "pass a model, choose another tradition, or omit tradition entirely"        )    body = {"messages": messages, "temperature": temperature,            "max_tokens": max_tokens, "tradition": tradition}    if model:        body["model"] = model    else:        body["auto_routing"] = True    body.update(extra or {})    response = requests.post(        url or GLOO_COMPLETIONS_URL,        headers={"Content-Type": "application/json",                 "Authorization": f"Bearer {gloo_token()}"},        json=body,        timeout=90,    )    response.raise_for_status()    return response.json()FACILITATOR_SYSTEM = (    "You are the Facilitator for Round, a small-group Scripture reading circle. "    "Before the group reads the passage, write two short pre-reading prompts. "    "Each is one sentence, plain language, no jargon, and answerable by someone "    "who has never read this passage. Do not summarise or interpret the passage. "    "Return them as a plain numbered list and nothing else.")if not HAS_GLOO:    print("No Gloo credentials — skipping live completion.")else:    raw = gloo_completion([        {"role": "system", "content": FACILITATOR_SYSTEM},        {"role": "user", "content": "Today's passage is Psalm 23. The circle is four people, mixed experience."},    ])    show("Pre-reading prompts (Facilitator agent)", raw["choices"][0]["message"]["content"])    print()    print("Routed to:", raw.get("model") or "(no model reported)")    print("Tokens:  prompt", raw.get("usage", {}).get("prompt_tokens"),          "· completion", raw.get("usage", {}).get("completion_tokens"))

---# Part 3 — Three findings that shaped the buildEverything above is API usage. This part is the engineering: three things thatwere not in any documentation, were found by running the APIs hard against realdevotional content, and each of which changed the code.## Finding 1 — a refusal can look exactly like a good answerGloo produces refusals in **two structurally different shapes**, both observedlive:**Shape 1 — the pre-routing guardrail.** A `200` in the normal completion shapecarrying a canned refusal as the content, and *no routing metadata*: `model`comes back empty and the Gloo-only fields (`provider`, `model_family`,`routing_mechanism`, `trace_id`) are absent. The tell is structural — no modelwas ever routed to.**Shape 2 — a refusal written by the routed model**, after Gloo's safety layerflagged the input *to* it. The response is completely normal: `model` ispopulated (e.g. `gloo-google-gemini-2.5-flash`), usage is reported. Only theprose gives it away.Shape 2 is the dangerous one. Without detection, it lands in `agent_logs` as`ok`, renders to the user as the agent's output, and — the reason this matters atall — **can be posted to their circle as their own words** when it happens on theTranslation agent's path.It was found on a prayer recast whose text quoted **Colossians 3:5** ("sexualimmorality, lust, and greed"). Scriptural vice language reads as toxicity to acontent classifier. That is the general lesson for anyone building on faithcontent: the Bible contains violence, sexuality, and vice by the chapter, andsafety infrastructure tuned on general web text will flag it.The detection markers below are **deliberately narrow**. Every entry is a phrasean assistant declining a request writes and a pastoral, devotional, or translatedcompletion does not. Weak markers like "I can't help with that" are excluded onpurpose — a circle member could write that sentence in a message, and theTranslation agent would then refuse to carry their words. The 400-characterwindow applies the same discipline: a refusal opens with its refusal, so a1,200-character prayer that happens to contain a matching phrase deep in its bodyis not one.And it is **not transient**: retrying the same text is refused identically everytime (verified), so callers must change the request or give up rather than spendanother call — which is why `isTransient()` excludes it from the retry path.

In [ ]:
MODEL_REFUSAL_MARKERS = [    r"flagged by an automated safety check",    r"\b(?:I'?m|I am) unable to (?:directly )?(?:assist|help|comply|fulfill|process)\b",    r"\bunable to fulfill (?:this|that|your)(?: specific)? request\b",    r"\bI cannot (?:assist|comply) with (?:this|that|your) request\b",    r"\b(?:violates|goes against) (?:our|the|my) (?:content |usage )?polic(?:y|ies)\b",    r"\bas an AI (?:language )?model\b",]REFUSAL_WINDOW_CHARS = 400   # a refusal opens with its refusaldef looks_like_model_refusal(content):    """Port of looksLikeModelRefusal() — shape-2 detection."""    opening = content[:REFUSAL_WINDOW_CHARS]    return any(re.search(marker, opening, re.IGNORECASE) for marker in MODEL_REFUSAL_MARKERS)def is_guardrail_shape_1(raw):    """Shape 1: a 200 with no model and no routing metadata — nothing was routed to."""    routing_fields = ("provider", "model_family", "routing_mechanism", "trace_id")    return not raw.get("model") and not any(raw.get(field) for field in routing_fields)cases = [    ("refusal (shape 2)", "I'm unable to fulfill this specific request. Your message was flagged by an automated safety check."),    ("real prayer",       "Father, we come to you tired. We have wanted things that did not want us back, and we ask you to put those wants to death in us, gently, and give us appetite for you instead."),    ("member message",    "Honestly I can't help with that one, I've read Psalm 23 a hundred times and it still gets me."),]for label, text in cases:    print(f"{label:<20} refusal={looks_like_model_refusal(text)}")

The third case is the one that justifies the narrowness. A circle member writing*"I can't help with that one"* in a heartfelt message must not be classified as arefusal — otherwise the Translation agent silently drops a person's words. Alooser marker list passes the first test and fails the product.## Finding 2 — scope retrieval by identity, not by similarityRound's Companion agent answers questions about the passage in front of you,grounded in public-domain commentary (Spurgeon's *Treasury of David*) uploaded tothe Gloo **Data Engine**. The obvious approach is the one-call groundedcompletions path. Measured against the live corpus, it surfaced roughly **7%** ofa Psalm's commentary.The reason is that server-side filtering does not exist on the Search endpoint:`producer_id`, `item_title`, `filters`, and `where` in the request body are**silently ignored** — accepted, no error, no effect. And `producer_id` comesback `null` on search results, so it cannot be used to filter client-side either.The working pattern is **over-fetch wide, then filter by `filename`**, which isthe only reliable scope key, and which the corpus makes exact by being one itemper Psalm. Sort survivors by `part` to restore reading order and pass them toCompletions V2. That recovers roughly **74%** of a Psalm's commentary.The deeper point is not the number. Scoping by item *identity* rather thansemantic similarity means **no chunk from a neighbouring Psalm can reach theprompt**, and zero survivors is a definitive *"the commentary does not coverthis"* rather than a weak inference from low similarity scores. For a Scriptureproduct, a grounded answer that quietly drifts one Psalm sideways is worse thanno answer.One deployment trap worth recording: a `403` from Search is almost always thetenant string. The normalised publisher name uses **underscores, not hyphens**(`scripture_commentary`), and the hyphenated form is rejected outright.

In [ ]:
def gloo_search(query, limit=100, certainty=0.0, tenant=None):    """Port of searchCorpus() — chunk-level retrieval with parent-item identity attached."""    tenant = tenant or GLOO_SEARCH_TENANT    if not tenant:        raise ValueError("GLOO_SEARCH_TENANT is not set and no tenant was passed")    response = requests.post(        GLOO_SEARCH_URL,        headers={"Content-Type": "application/json",                 "Authorization": f"Bearer {gloo_token()}"},        json={"collection": GLOO_COLLECTION, "tenant": tenant,              "query": query, "limit": limit, "certainty": certainty},        timeout=60,    )    if response.status_code == 403:        raise RuntimeError("403 from Search — check the tenant string uses underscores, not hyphens")    response.raise_for_status()    return [{        "filename":  hit.get("properties", {}).get("filename"),        "itemTitle": hit.get("properties", {}).get("item_title"),        "part":      hit.get("properties", {}).get("part"),        "snippet":   hit.get("properties", {}).get("snippet") or "",        "certainty": hit.get("metadata", {}).get("certainty"),    } for hit in response.json().get("data", [])]def scoped_chunks(query, filename):    """Over-fetch wide, filter by filename (identity), sort by part (reading order)."""    hits = gloo_search(query, limit=100, certainty=0.0)    kept = [hit for hit in hits if hit["filename"] == filename]    kept.sort(key=lambda hit: hit["part"] if hit["part"] is not None else 0)    return hits, keptif not (HAS_GLOO and GLOO_SEARCH_TENANT):    print("No Gloo credentials or GLOO_SEARCH_TENANT — skipping live retrieval.")else:    # Name passages in HUMAN form. USFM ("PSA.23") matches nothing in the corpus.    result = live("Search", lambda: scoped_chunks("Psalm 23 the Lord is my shepherd", "psalm-023.md"))    everything, scoped = result if result else ([], [])    print(f"retrieved wide : {len(everything)} chunks")    print(f"after filename : {len(scoped)} chunks from psalm-023.md")    print(f"discarded      : {len(everything) - len(scoped)} chunks from other Psalms\n")    if scoped:        show(f"part {scoped[0]['part']} — {scoped[0]['itemTitle']}",             re.sub(r"\s+", " ", scoped[0]["snippet"])[:600] + "...")

### The grounded path, and where it is still the right toolThe one-call grounded endpoint is not abandoned — it is used where breadth beatsscope, and it returns citations the app renders as source attribution. Twothings it needs that are easy to miss: `rag_publisher` is required, and`include_citations` defaults to **false**, which would silently drop thecitations the Context Agent needs both for its scope guard and for attribution.`sources_limit` must be an integer 1–10.

In [ ]:
def gloo_grounded(messages, rag_publisher, sources_limit=5, **kwargs):    """Port of groundedCompletion() — RAG path with citations forced on."""    if not 1 <= sources_limit <= 10:        raise ValueError(f"sources_limit must be an integer between 1 and 10, got {sources_limit}")    return gloo_completion(        messages,        url=GLOO_GROUNDED_URL,        extra={"rag_publisher": rag_publisher,               "sources_limit": sources_limit,               "include_citations": True},   # Gloo defaults this to False        **kwargs,    )rag_publisher = secret("GLOO_RAG_PUBLISHER")if not (HAS_GLOO and rag_publisher):    print("No Gloo credentials or GLOO_RAG_PUBLISHER — skipping grounded call.")elif "_" in rag_publisher:    # rag_publisher is the publisher's DISPLAY NAME (hyphens); the underscore    # form is the Search tenant. Sending the tenant here returns 403.    print("GLOO_RAG_PUBLISHER looks like the tenant form (underscores) — "          "grounded completions need the hyphenated display name.")else:    raw = live("Grounded completion", lambda: gloo_grounded(        [{"role": "system", "content": "You are Round's Companion agent. Answer from the supplied sources only. If they do not cover the question, say so plainly."},         {"role": "user",   "content": "In Psalm 23, what does the rod and staff comfort actually refer to?"}],        rag_publisher=rag_publisher,    ))    if raw:        answer = raw["choices"][0]["message"]["content"]        show("Companion agent (grounded)", answer)        print("\nrefusal detected:", looks_like_model_refusal(answer))        print("sources_returned:", raw.get("sources_returned"))        for citation in (raw.get("citations") or [])[:3]:            print(" ·", citation.get("item_title"), "—", citation.get("publisher"))

## Finding 3 — make the audit trail impossible to skipThe brief requires that all agent outputs are logged. The way to fail thatrequirement is to make logging a thing each agent remembers to do.Round's `chatCompletion()` writes the `agent_logs` row itself, **before itreturns** — on success *and* on failure, including guardrail refusals andexhausted retries. Every call therefore carries `agentName`, the model thatactually served it, token usage, latency, outcome, and a 500-character`output_preview` (a reference, never the full text). Because the client is theonly door to a model, the rule holds *by construction*: an agent cannot make anunlogged call without adding a second HTTP client to the repo, which is exactlythe thing the `grep` in rule 1 would catch.Two supporting details:- `logAgentNote()` records non-model events on the same trail — for example how  many retrieved chunks survived the filename filter, a number that exists  nowhere else once filtering moved client-side. It carries a note, never  content, and cannot fail a request.- `writeAgentLog()` swallows its own errors and logs loudly to the server  console. Logging must never turn a delivered completion into a user-facing  failure.Retries are 1 initial + 2, exponential from 500ms, and **only on transienterrors** (429 and 5xx). Guardrail refusals are excluded because they aredeterministic — retrying spends a call to receive the identical refusal. A 401clears the cached token so the next attempt re-authenticates.

In [ ]:
AGENT_LOG_COLUMNS = [    "agent_name",       # facilitator · companion · circle-bot · prayer · reminder · demo-refresh    "model",            # what auto-routing actually chose; null on search calls    "prompt_tokens",    "completion_tokens",    "latency_ms",    "status",           # ok · guardrail · error    "output_preview",   # first 500 chars — a reference, never the full text    "created_at",]print("agent_logs — one row per Gloo call, written before the call returns:\n")for column in AGENT_LOG_COLUMNS:    print("  ·", column)print("\nSchema: src/db/schema.ts · writer: writeAgentLog() in src/lib/gloo.ts")

---# Where to look in the repo| Concern | File ||---|---|| Gloo gateway, auth, retries, refusal detection, logging | `src/lib/gloo.ts` || YouVersion adapter — passages, versions, index, highlights | `src/lib/youversion.ts` || Version catalogue + licensing fallbacks | `src/config/bible-versions.ts` || Sign in with YouVersion (OAuth) | `src/lib/auth.ts` || Facilitator — pre-reading prompts | `src/lib/pre-reading.ts` || Facilitator — conversation starters, lesson summary | `src/lib/post-reading.ts` || Companion — passage Q&A over the corpus | `src/lib/companion.ts`, `src/lib/corpus.ts`, `src/lib/passage-qa.ts` || Escalation screen before anything reaches a circle | `src/lib/escalation.ts` || Seeded AI circle members | `src/lib/circle-bot.ts` || Prayer generation from the reader's own reading | `src/lib/prayer-context.ts`, `src/lib/prayer-requests.ts` || AI-generated custom plans (validated against YouVersion) | `src/lib/plan-generation.ts` || Circle matching | `src/lib/matching.ts` || Translation of circle messages | `src/lib/translation.ts` || No-account demo path | `src/lib/anon-session.ts`, `src/lib/demo-seed.ts` || Corpus upload + chunking scripts | `scripts/upload-corpus.mts`, `scripts/split-psalms-commentary.mts` |**Try the product:** https://yv-gloo-chalange.vercel.app — tap **Instant Access**.No sign-up, every AI call live, every passage fetched live.**Watch the film:** https://www.youtube.com/watch?v=YzcBW4xfzeM*Anna and the circle members in the video are dramatizations performed by actors.*